In [1]:
!git clone https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git
%cd SMILES-HALLUCINATION-DETECTION
!pip install -r requirements.txt

Cloning into 'SMILES-HALLUCINATION-DETECTION'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 38 (delta 16), reused 10 (delta 10), pack-reused 12 (from 1)
Receiving objects: 100% (38/38), 451.08 KiB | 2.25 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/SMILES-HALLUCINATION-DETECTION


### Посмотрим какие результаты мы получим с дефолтными версиями кодов:

In [ ]:
# default codes
!python solution.py

Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontractor.<|endoftext|>

── label : 1  (hall

### Теперь попробуем кросс-валидацию:

In [ ]:
# k-fold
!python solution.py

Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontractor.<|endoftext|>

── label : 1  (hall

## Попробуем PCA + MLP

In [ ]:
# PCA + MLP
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontr

## Article: https://arxiv.org/pdf/2604.06277

### M0: ProbeMLP

```python

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


class ProbeMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class HallucinationProbe(nn.Module):

    def __init__(self):
        super().__init__()

        self._net = None
        self._scaler = StandardScaler()
        self._threshold = 0.5

    def _build_network(self, input_dim):
        self._net = ProbeMLP(input_dim)

    def forward(self, x):
        return self._net(x)

    def fit(self, X, y):

        X = self._scaler.fit_transform(X)

        self._build_network(X.shape[1])

        X_t = torch.from_numpy(X).float()
        y_t = torch.from_numpy(y.astype(np.float32))

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos

        pos_weight = torch.tensor([
            n_neg / max(n_pos, 1)
        ]).float()

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=1e-3,
            weight_decay=1e-2,
        )

        self.train()

        for _ in range(100):

            optimizer.zero_grad()

            logits = self(X_t)

            loss = criterion(logits, y_t)

            loss.backward()

            optimizer.step()

        self.eval()

        return self

    def fit_hyperparameters(self, X_val, y_val):

        probs = self.predict_proba(X_val)[:, 1]

        best_t = 0.5
        best_f1 = -1

        for t in np.linspace(0.1, 0.9, 81):

            preds = (probs >= t).astype(int)

            score = f1_score(y_val, preds)

            if score > best_f1:
                best_f1 = score
                best_t = t

        self._threshold = best_t

        return self

    def predict(self, X):
        return (
            self.predict_proba(X)[:, 1] >= self._threshold
        ).astype(int)

    def predict_proba(self, X):

        X = self._scaler.transform(X)

        X_t = torch.from_numpy(X).float()

        with torch.no_grad():
            logits = self(X_t)
            probs = torch.sigmoid(logits).numpy()

        return np.stack([1 - probs, probs], axis=1)
```

In [2]:
# M0
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

[Errno 2] No such file or directory: 'SMILES-HALLUCINATION-DETECTION'
/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ──────────────────────────────

### M1: LayerWiseMLP

```python
from __future__ import annotations

import torch
import torch.nn.functional as F


N_LAST_LAYERS = 8


def aggregate(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    M1-compatible aggregation.

    Output:
        (8 * hidden_dim,)
    """

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]
    hs = hs[-N_LAST_LAYERS:]
    pooled = hs.mean(dim=1)
    feature = pooled.reshape(-1)

    return feature.float()


def extract_geometric_features(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]

    hs = hs[-N_LAST_LAYERS:]

    pooled = hs.mean(dim=1)

    cosines = []

    for i in range(1, pooled.size(0)):

        c = F.cosine_similarity(
            pooled[i].unsqueeze(0),
            pooled[i - 1].unsqueeze(0),
            dim=-1,
        )

        cosines.append(c)

    cosines = torch.cat(cosines)

    features = torch.tensor(
        [
            norms.mean(),
            norms.std(),
            norms.max(),
            norms.min(),

            cosines.mean(),
            cosines.std(),
            cosines.max(),
            cosines.min(),

            float(real_len),
        ],
        dtype=torch.float32,
    )

    return features


def aggregation_and_feature_extraction(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
    use_geometric: bool = False,
) -> torch.Tensor:

    agg = aggregate(
        hidden_states,
        attention_mask,
    )

    if use_geometric:

        geo = extract_geometric_features(
            hidden_states,
            attention_mask,
        )

        return torch.cat([agg, geo], dim=0)

    return agg
```


In [ ]:
# M1
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontr

### M2: CrossLayerTransformer

```python
"""
aggregation.py
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


N_LAST_LAYERS = 8


def aggregate(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Returns:
        flattened layer-wise representation

    Shape:
        (8 * 896,)
    """

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]
    hs = hs[-N_LAST_LAYERS:]
    pooled = hs.mean(dim=1)

    feature = pooled.reshape(-1)

    return feature.float()
    

def extract_geometric_features(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]

    hs = hs[-N_LAST_LAYERS:]

    pooled = hs.mean(dim=1)

    norms = pooled.norm(dim=-1)

    cosines = []

    for i in range(1, pooled.size(0)):

        c = F.cosine_similarity(
            pooled[i].unsqueeze(0),
            pooled[i - 1].unsqueeze(0),
            dim=-1,
        )

        cosines.append(c)

    cosines = torch.cat(cosines)

    features = torch.tensor(
        [
            norms.mean(),
            norms.std(),
            norms.max(),
            norms.min(),

            cosines.mean(),
            cosines.std(),
            cosines.max(),
            cosines.min(),

            float(real_len),
        ],
        dtype=torch.float32,
    )

    return features



def aggregation_and_feature_extraction(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
    use_geometric: bool = False,
) -> torch.Tensor:

    agg = aggregate(
        hidden_states,
        attention_mask,
    )

    if use_geometric:

        geo = extract_geometric_features(
            hidden_states,
            attention_mask,
        )

        return torch.cat([agg, geo], dim=0)

    return agg
```


```python

# probe.py for M2


from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


N_LAST_LAYERS = 8
HIDDEN_DIM = 896


class CrossLayerTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(HIDDEN_DIM, 256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=512,
            dropout=0.2,
            activation="gelu",
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1),
        )

    def forward(self, x):

        batch = x.shape[0]

        x = x.view(batch, N_LAST_LAYERS, HIDDEN_DIM)

        x = self.input_proj(x)

        x = self.transformer(x)

        x = x.mean(dim=1)

        logits = self.classifier(x)

        return logits.squeeze(-1)




class HallucinationProbe(nn.Module):

    def __init__(self):
        super().__init__()

        self._net = CrossLayerTransformer()

        self._scaler = StandardScaler()

        self._threshold = 0.5

    def forward(self, x):

        return self._net(x)

    def fit(self, X, y):

        X_scaled = self._scaler.fit_transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        y_t = torch.from_numpy(
            y.astype(np.float32)
        )

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos

        pos_weight = torch.tensor(
            [n_neg / max(n_pos, 1)],
            dtype=torch.float32,
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=5e-4,
            weight_decay=1e-2,
        )

        self.train()

        best_loss = float("inf")

        patience = 20
        patience_counter = 0

        for epoch in range(200):

            optimizer.zero_grad()

            logits = self(X_t)

            loss = criterion(logits, y_t)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.parameters(),
                1.0,
            )

            optimizer.step()

            current_loss = loss.item()

            if current_loss < best_loss:

                best_loss = current_loss
                patience_counter = 0

            else:

                patience_counter += 1

            if patience_counter >= patience:
                break

        self.eval()

        return self

    def fit_hyperparameters(self, X_val, y_val):

        probs = self.predict_proba(X_val)[:, 1]

        best_threshold = 0.5
        best_f1 = -1

        for t in np.linspace(0.1, 0.9, 81):

            preds = (probs >= t).astype(int)

            score = f1_score(
                y_val,
                preds,
                zero_division=0,
            )

            if score > best_f1:

                best_f1 = score
                best_threshold = t

        self._threshold = best_threshold

        return self

    def predict(self, X):

        probs = self.predict_proba(X)[:, 1]

        return (
            probs >= self._threshold
        ).astype(int)

    def predict_proba(self, X):

        X_scaled = self._scaler.transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        with torch.no_grad():

            logits = self(X_t)

            probs = torch.sigmoid(logits).numpy()

        return np.stack(
            [1 - probs, probs],
            axis=1,
        )
```

In [ ]:
# M2
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontr

### M3: HierarchicalTransformer

```python
"""
aggregation.py
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


N_LAST_LAYERS = 8


def aggregate(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Returns:
        flattened layer-wise representation

    Shape:
        (8 * 896,)
    """

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]
    hs = hs[-N_LAST_LAYERS:]
    pooled = hs.mean(dim=1)

    feature = pooled.reshape(-1)

    return feature.float()
    

def extract_geometric_features(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]

    hs = hs[-N_LAST_LAYERS:]

    pooled = hs.mean(dim=1)

    norms = pooled.norm(dim=-1)

    cosines = []

    for i in range(1, pooled.size(0)):

        c = F.cosine_similarity(
            pooled[i].unsqueeze(0),
            pooled[i - 1].unsqueeze(0),
            dim=-1,
        )

        cosines.append(c)

    cosines = torch.cat(cosines)

    features = torch.tensor(
        [
            norms.mean(),
            norms.std(),
            norms.max(),
            norms.min(),

            cosines.mean(),
            cosines.std(),
            cosines.max(),
            cosines.min(),

            float(real_len),
        ],
        dtype=torch.float32,
    )

    return features



def aggregation_and_feature_extraction(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
    use_geometric: bool = False,
) -> torch.Tensor:

    agg = aggregate(
        hidden_states,
        attention_mask,
    )

    if use_geometric:

        geo = extract_geometric_features(
            hidden_states,
            attention_mask,
        )

        return torch.cat([agg, geo], dim=0)

    return agg
```


```python

# probe.py for M3

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


N_LAST_LAYERS = 8
HIDDEN_DIM = 896


class HierarchicalTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # layer encoder
        self.layer_encoder = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 256),
            nn.GELU(),
            nn.Dropout(0.2),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            dim_feedforward=512,
            dropout=0.2,
            activation="gelu",
            batch_first=True,
        )

        self.cross_layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1),
        )

    def forward(self, x):

        batch = x.shape[0]

        x = x.view(batch, N_LAST_LAYERS, HIDDEN_DIM)

        x = self.layer_encoder(x)

        x = self.cross_layer_transformer(x)

        x = x.mean(dim=1)

        logits = self.classifier(x)

        return logits.squeeze(-1)


class HallucinationProbe(nn.Module):

    def __init__(self):
        super().__init__()

        self._net = HierarchicalTransformer()

        self._scaler = StandardScaler()

        self._threshold = 0.5

    def forward(self, x):

        return self._net(x)

    def fit(self, X, y):

        X_scaled = self._scaler.fit_transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        y_t = torch.from_numpy(
            y.astype(np.float32)
        )

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos

        pos_weight = torch.tensor(
            [n_neg / max(n_pos, 1)],
            dtype=torch.float32,
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=5e-4,
            weight_decay=1e-2,
        )

        self.train()

        best_loss = float("inf")

        patience = 20
        patience_counter = 0

        for epoch in range(200):

            optimizer.zero_grad()

            logits = self(X_t)

            loss = criterion(logits, y_t)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.parameters(),
                1.0,
            )

            optimizer.step()

            current_loss = loss.item()

            if current_loss < best_loss:

                best_loss = current_loss
                patience_counter = 0

            else:

                patience_counter += 1

            if patience_counter >= patience:
                break

        self.eval()

        return self

    def fit_hyperparameters(self, X_val, y_val):

        probs = self.predict_proba(X_val)[:, 1]

        best_threshold = 0.5
        best_f1 = -1

        for t in np.linspace(0.1, 0.9, 81):

            preds = (probs >= t).astype(int)

            score = f1_score(
                y_val,
                preds,
                zero_division=0,
            )

            if score > best_f1:

                best_f1 = score
                best_threshold = t

        self._threshold = best_threshold

        return self

    def predict(self, X):

        probs = self.predict_proba(X)[:, 1]

        return (
            probs >= self._threshold
        ).astype(int)

    def predict_proba(self, X):

        X_scaled = self._scaler.transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        with torch.no_grad():

            logits = self(X_t)

            probs = torch.sigmoid(logits).numpy()

        return np.stack(
            [1 - probs, probs],
            axis=1,
        )
```

In [ ]:
# M3
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontr

### M4: CrossLayerAttentionTransformerV2

```python
"""
aggregation.py
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


N_LAST_LAYERS = 8


def aggregate(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Returns:
        flattened layer-wise representation

    Shape:
        (8 * 896,)
    """

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]
    hs = hs[-N_LAST_LAYERS:]
    pooled = hs.mean(dim=1)

    feature = pooled.reshape(-1)

    return feature.float()
    

def extract_geometric_features(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:

    real_len = int(attention_mask.sum().item())

    hs = hidden_states[:, :real_len]

    hs = hs[-N_LAST_LAYERS:]

    pooled = hs.mean(dim=1)

    norms = pooled.norm(dim=-1)

    cosines = []

    for i in range(1, pooled.size(0)):

        c = F.cosine_similarity(
            pooled[i].unsqueeze(0),
            pooled[i - 1].unsqueeze(0),
            dim=-1,
        )

        cosines.append(c)

    cosines = torch.cat(cosines)

    features = torch.tensor(
        [
            norms.mean(),
            norms.std(),
            norms.max(),
            norms.min(),

            cosines.mean(),
            cosines.std(),
            cosines.max(),
            cosines.min(),

            float(real_len),
        ],
        dtype=torch.float32,
    )

    return features



def aggregation_and_feature_extraction(
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
    use_geometric: bool = False,
) -> torch.Tensor:

    agg = aggregate(
        hidden_states,
        attention_mask,
    )

    if use_geometric:

        geo = extract_geometric_features(
            hidden_states,
            attention_mask,
        )

        return torch.cat([agg, geo], dim=0)

    return agg
```


```python

# probe.py for M4


from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


N_LAST_LAYERS = 8
HIDDEN_DIM = 896


class CrossLayerAttentionTransformerV2(nn.Module):

    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(HIDDEN_DIM, 256)

        self.attention = nn.MultiheadAttention(
            embed_dim=256,
            num_heads=8,
            dropout=0.2,
            batch_first=True,
        )

        self.ffn = nn.Sequential(
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
        )

        self.norm1 = nn.LayerNorm(256)
        self.norm2 = nn.LayerNorm(256)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1),
        )

    def forward(self, x):

        batch = x.shape[0]

        x = x.view(batch, N_LAST_LAYERS, HIDDEN_DIM)

        x = self.input_proj(x)

        attn_out, _ = self.attention(x, x, x)

        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)

        x = self.norm2(x + ffn_out)

        x = x.mean(dim=1)

        logits = self.classifier(x)

        return logits.squeeze(-1)


class HallucinationProbe(nn.Module):

    def __init__(self):
        super().__init__()

        self._net = CrossLayerAttentionTransformerV2()

        self._scaler = StandardScaler()

        self._threshold = 0.5

    def forward(self, x):

        return self._net(x)

    def fit(self, X, y):

        X_scaled = self._scaler.fit_transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        y_t = torch.from_numpy(
            y.astype(np.float32)
        )

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos

        pos_weight = torch.tensor(
            [n_neg / max(n_pos, 1)],
            dtype=torch.float32,
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=5e-4,
            weight_decay=1e-2,
        )

        self.train()

        best_loss = float("inf")

        patience = 20
        patience_counter = 0

        for epoch in range(200):

            optimizer.zero_grad()

            logits = self(X_t)

            loss = criterion(logits, y_t)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.parameters(),
                1.0,
            )

            optimizer.step()

            current_loss = loss.item()

            if current_loss < best_loss:

                best_loss = current_loss
                patience_counter = 0

            else:

                patience_counter += 1

            if patience_counter >= patience:
                break

        self.eval()

        return self

    def fit_hyperparameters(self, X_val, y_val):

        probs = self.predict_proba(X_val)[:, 1]

        best_threshold = 0.5
        best_f1 = -1

        for t in np.linspace(0.1, 0.9, 81):

            preds = (probs >= t).astype(int)

            score = f1_score(
                y_val,
                preds,
                zero_division=0,
            )

            if score > best_f1:

                best_f1 = score
                best_threshold = t

        self._threshold = best_threshold

        return self

    def predict(self, X):

        probs = self.predict_proba(X)[:, 1]

        return (
            probs >= self._threshold
        ).astype(int)

    def predict_proba(self, X):

        X_scaled = self._scaler.transform(X)

        X_t = torch.from_numpy(X_scaled).float()

        with torch.no_grad():

            logits = self(X_t)

            probs = torch.sigmoid(logits).numpy()

        return np.stack(
            [1 - probs, probs],
            axis=1,
        )
```

In [ ]:
# M4
%cd SMILES-HALLUCINATION-DETECTION
!python solution.py

/content/SMILES-HALLUCINATION-DETECTION
Device       : cuda
Data         : ./data/dataset.csv
Max length   : 512 tokens
Geometric feats: False
Loaded 689 samples  (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

── prompt (first 500 chars) ──────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

── response (first 300 chars) ───────────────────────────────
An architect or engineer has a direct relationship with the subcontr